In [ ]:
import os
import pandas as pd
# pyrefly: ignore [missing-import]
import numpy as np
import traceback
import warnings
import time
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score

# Suppress convergence warnings during HP tuning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

from config import DATASETS, N_SEED_SETS, N_ITER, RESULTS_DIR, N_SPLITS
from dataset import load_dataset, get_feature_columns, get_train_test_splits, get_generation_seed_sets
from models import get_models_and_params, tune_hyperparameters
import augmentations
from utils import plot_performance_comparison, save_results_table

def extract_timeseries_features(df, feature_cols):
    """
    Helper function to reshape dataframe into (samples, timesteps, features).
    """
    grouped = df.groupby('Measurement_Number')
    
    X = []
    y = []
    for name, group in grouped:
        # Sort by Time to keep sequential order
        group = group.sort_values('Time')
        # Keep as (timesteps, features) to preserve 3D shape for augmentations
        features = group[feature_cols].values
        label = group['Label'].iloc[0]
        
        X.append(features)
        y.append(label)
        
    return np.array(X), np.array(y)

def run_experiment(dataset_name):
    print(f"\n{'='*50}\nRunning experiment for {dataset_name}\n{'='*50}")
    
    try:
        df = load_dataset(dataset_name)
    except Exception as e:
        print(f"Could not load {dataset_name}: {e}")
        return
        
    feature_cols = get_feature_columns(df)
    
    # Dictionary to store all results
    all_results = []
    
    # Get the N folds
    fold_idx = 1
    for full_train_idx, test_idx in get_train_test_splits(df):
        print(f"\n--- Fold {fold_idx}/{N_SPLITS} ---")
        
        test_df = df.iloc[test_idx]
        X_test, y_test = extract_timeseries_features(test_df, feature_cols)
        
        # In each fold, we get generation seed sets (1->20 measurement sets per label)
        # Testing with [1, 3, 5, 7, 10, 15, 18, 20] sets per label
        for n_seeds in [1, 3, 5, 7, 10, 15, 18, 20]:
            print(f"\n  >> Seed sets per label: {n_seeds}")
            seed_indices = get_generation_seed_sets(df, full_train_idx, num_sets=n_seeds)
            seed_df = df.iloc[seed_indices]
            
            X_seed, y_seed = extract_timeseries_features(seed_df, feature_cols)
            
            models_dict = get_models_and_params()
            
            for model_name, model_info in models_dict.items():
                print(f"    Evaluating Model: {model_name}")
                base_model = model_info['model']
                params = model_info['params']
                
                is_dl_model = model_name in ['GRU', 'RNN']
                
                # Prepare point-level data for Scikit-Learn models vs sequence-level for DL
                if not is_dl_model:
                    X_seed_model = X_seed.reshape(-1, X_seed.shape[2])
                    y_seed_model = np.repeat(y_seed, X_seed.shape[1])
                    groups_seed_model = np.repeat(np.arange(X_seed.shape[0]), X_seed.shape[1])
                    X_test_model = X_test.reshape(-1, X_test.shape[2])
                    y_test_model = np.repeat(y_test, X_test.shape[1])
                else:
                    X_seed_model = X_seed
                    y_seed_model = y_seed
                    groups_seed_model = None
                    X_test_model = X_test
                    y_test_model = y_test
                
                # --- 1. Baseline (Train on Seed Set, Evaluate on Test Set) ---
                try:
                    start_time = time.time()
                    base_model.fit(X_seed_model, y_seed_model)
                    y_pred_base = base_model.predict(X_test_model)
                    exec_time = time.time() - start_time
                    
                    acc_base = accuracy_score(y_test_model, y_pred_base)
                    f1_base = f1_score(y_test_model, y_pred_base, average='weighted')
                    params_str = str(base_model.get_params()) if hasattr(base_model, 'get_params') else "N/A"
                    res_dict = {
                        'Dataset': dataset_name, 'Fold': fold_idx, 'SeedsPerLabel': n_seeds,
                        'ScaleFactor': 1,
                        'Model': model_name, 'Method': 'Baseline', 'Accuracy': acc_base,
                        'F1_Score': f1_base, 'Hyperparameters': params_str, 'ExecutionTime_Sec': exec_time
                    }
                    all_results.append(res_dict)
                    print(f"      -> [Baseline] Acc: {acc_base:.4f}, F1: {f1_base:.4f}, Time: {exec_time:.2f}s")
                    save_results_table(pd.DataFrame(all_results), dataset_name)
                except Exception as e:
                    print(f"      [!] Baseline failed for {model_name}: {e}")
                    continue
                
                # --- 2. HP Tuning (on Seed Set) ---
                if model_name not in ['GRU', 'RNN']:
                    print(f"      HP Tuning {model_name}...")
                    try:
                        if n_seeds < 2:
                            # Fallback to standard CV (may leak) when there's not enough measurements for grouped CV
                            best_model, best_params = tune_hyperparameters(
                                base_model, params, X_seed_model, y_seed_model, 
                                groups=None, n_iter=N_ITER, cv=2
                            )
                        else:
                            best_model, best_params = tune_hyperparameters(
                                base_model, params, X_seed_model, y_seed_model, 
                                groups=groups_seed_model, n_iter=N_ITER, cv=2
                            )
                    except Exception as e:
                        print(f"      [!] HP Tuning failed, using base model: {e}")
                        best_model = base_model
                else:
                    best_model = base_model # Deep learning skeleton
                    
                # --- 2.5 Tuned Baseline (Evaluate tuned model on Test Set) ---
                try:
                    start_time = time.time()
                    y_pred_tuned = best_model.predict(X_test_model)
                    exec_time = time.time() - start_time
                    
                    acc_tuned = accuracy_score(y_test_model, y_pred_tuned)
                    f1_tuned = f1_score(y_test_model, y_pred_tuned, average='weighted')
                    params_str = str(best_model.get_params()) if hasattr(best_model, 'get_params') else "N/A"
                    res_dict = {
                        'Dataset': dataset_name, 'Fold': fold_idx, 'SeedsPerLabel': n_seeds,
                        'ScaleFactor': 1,
                        'Model': model_name, 'Method': 'Tuned_Baseline', 'Accuracy': acc_tuned,
                        'F1_Score': f1_tuned, 'Hyperparameters': params_str, 'ExecutionTime_Sec': exec_time
                    }
                    all_results.append(res_dict)
                    print(f"      -> [Tuned_Baseline] Acc: {acc_tuned:.4f}, F1: {f1_tuned:.4f}, Time: {exec_time:.2f}s")
                    save_results_table(pd.DataFrame(all_results), dataset_name)
                except Exception as e:
                    print(f"      [!] Tuned Baseline failed for {model_name}: {e}")
                
                # --- 3. Generation Methods ---
                generation_methods = {
                    'Jittering': lambda X, y: (augmentations.jitter(X), y),
                    'Scaling': lambda X, y: (augmentations.scaling(X), y),
                    'Permutation': lambda X, y: (augmentations.permutation(X), y),
                    'Magnitude_Warping': lambda X, y: (augmentations.magnitude_warping(X), y),
                    'Time_Warping': lambda X, y: (augmentations.time_warping(X), y),
                    'Window_Slicing': lambda X, y: (augmentations.window_slicing(X), y),
                    'SMOTE': augmentations.apply_smote,
                    'ADASYN': augmentations.apply_adasyn,
                    'Mixup': augmentations.mixup,
                    'CTGAN': augmentations.apply_ctgan,
                    'VAE': augmentations.apply_vae
                }
                
                for gen_name, gen_func in generation_methods.items():
                    # Condition 1: SMOTE and ADASYN only when seed size >= 5
                    if gen_name in ['SMOTE', 'ADASYN'] and n_seeds < 5:
                        continue
                    # Condition 2: CTGAN and VAEs only when seed size > 10
                    if gen_name in ['CTGAN', 'VAE'] and n_seeds <= 10:
                        continue
                    
                    for scale_factor in [2, 5, 10, 20]:
                        print(f"      Generating {scale_factor}x synthetic data with {gen_name}...")
                        try:
                            # For SMOTE/ADASYN, pass scale_factor
                            if gen_name in ['SMOTE', 'ADASYN']:
                                X_syn, y_syn = gen_func(X_seed, y_seed, scale_factor=scale_factor)  # type: ignore
                            else:
                                # For other augmentations, generate multiple times and stack
                                X_syn_list = [X_seed]
                                y_syn_list = [y_seed]
                                for _ in range(scale_factor - 1):
                                    X_aug, y_aug = gen_func(X_seed, y_seed)
                                    X_syn_list.append(X_aug)
                                    y_syn_list.append(y_aug)
                                X_syn = np.vstack(X_syn_list)
                                y_syn = np.hstack(y_syn_list)
                            
                            # --- 4. Evaluate Synthetic dataset with HP-tuned model ---
                            if not is_dl_model:
                                X_syn_model = X_syn.reshape(-1, X_syn.shape[2])
                                y_syn_model = np.repeat(y_syn, X_syn.shape[1])
                            else:
                                X_syn_model = X_syn
                                y_syn_model = y_syn
                                
                            start_time = time.time()
                            best_model.fit(X_syn_model, y_syn_model)
                            y_pred_syn = best_model.predict(X_test_model)
                            exec_time = time.time() - start_time
                            
                            acc_syn = accuracy_score(y_test_model, y_pred_syn)
                            f1_syn = f1_score(y_test_model, y_pred_syn, average='weighted')
                            params_str = str(best_model.get_params()) if hasattr(best_model, 'get_params') else "N/A"
                            
                            res_dict = {
                                'Dataset': dataset_name, 'Fold': fold_idx, 'SeedsPerLabel': n_seeds,
                                'ScaleFactor': scale_factor,
                                'Model': model_name, 'Method': gen_name, 'Accuracy': acc_syn,
                                'F1_Score': f1_syn, 'Hyperparameters': params_str, 'ExecutionTime_Sec': exec_time
                            }
                            all_results.append(res_dict)
                            print(f"      -> [{gen_name} @ {scale_factor}x] Acc: {acc_syn:.4f}, F1: {f1_syn:.4f}, Time: {exec_time:.2f}s")
                            save_results_table(pd.DataFrame(all_results), dataset_name)
                        except Exception as e:
                            print(f"      [!] Failed generation with {gen_name} at {scale_factor}x: {e}")
                        
        fold_idx += 1
        
        # NOTE: For testing purposes, we break after 1 fold. 
        # Remove the break statement below to run all folds.
        print(f"\n[!] Breaking after 1 fold for demonstration. Remove 'break' in experiment.py to run full {N_SPLITS}-folds.")
        # break 
        
    # Summarize Results
    results_df = pd.DataFrame(all_results)
    save_results_table(results_df, dataset_name)
    
    # Plot average across folds for maximum seed size (e.g., 20)
    if not results_df.empty:
        # Choose a specific scale factor for plotting to avoid averaging across all scales
        plot_df = results_df[(results_df['SeedsPerLabel'] == 20) & (results_df['ScaleFactor'] == 10)].groupby(['Method', 'Model'])['Accuracy'].mean().reset_index()
        plot_performance_comparison(plot_df, dataset_name)

if __name__ == "__main__":
    # Ensure results dir exists
    os.makedirs(RESULTS_DIR, exist_ok=True)
    
    # Run experiment for all datasets
    for d_name in DATASETS.keys():
        run_experiment(d_name)
        # break # uncomment to run only 1 dataset for quick testing



Running experiment for Coffee

--- Fold 1/10 ---

  >> Seed sets per label: 1
    Evaluating Model: KNN
      -> [Baseline] Acc: 0.4922, F1: 0.4958, Time: 0.01s
      [+] Saved to Coffee_results.csv
      HP Tuning KNN...
Fitting 2 folds for each of 100 candidates, totalling 200 fits
      -> [Tuned_Baseline] Acc: 0.4922, F1: 0.4962, Time: 0.02s
      [+] Saved to Coffee_results.csv
      Generating 2x synthetic data with Jittering...
      -> [Jittering @ 2x] Acc: 0.4922, F1: 0.4961, Time: 0.02s
      [+] Saved to Coffee_results.csv
      Generating 5x synthetic data with Jittering...
      -> [Jittering @ 5x] Acc: 0.4922, F1: 0.4961, Time: 0.04s
      [+] Saved to Coffee_results.csv
      Generating 10x synthetic data with Jittering...
      -> [Jittering @ 10x] Acc: 0.4922, F1: 0.4961, Time: 0.05s
      [+] Saved to Coffee_results.csv
      Generating 20x synthetic data with Jittering...
      -> [Jittering @ 20x] Acc: 0.4928, F1: 0.4963, Time: 0.08s
      [+] Saved to Coffee_resul

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in versio

      -> [Tuned_Baseline] Acc: 0.4711, F1: 0.4375, Time: 0.00s
      [+] Saved to Coffee_results.csv
      Generating 2x synthetic data with Jittering...
      -> [Jittering @ 2x] Acc: 0.4556, F1: 0.4251, Time: 0.06s
      [+] Saved to Coffee_results.csv
      Generating 5x synthetic data with Jittering...
      -> [Jittering @ 5x] Acc: 0.4556, F1: 0.4268, Time: 0.92s
      [+] Saved to Coffee_results.csv
      Generating 10x synthetic data with Jittering...
      -> [Jittering @ 10x] Acc: 0.4533, F1: 0.4258, Time: 0.14s
      [+] Saved to Coffee_results.csv
      Generating 20x synthetic data with Jittering...


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> [Jittering @ 20x] Acc: 0.4550, F1: 0.4332, Time: 0.14s
      [+] Saved to Coffee_results.csv
      Generating 2x synthetic data with Scaling...
      -> [Scaling @ 2x] Acc: 0.4583, F1: 0.4375, Time: 0.19s
      [+] Saved to Coffee_results.csv
      Generating 5x synthetic data with Scaling...


...

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> [Window_Slicing @ 20x] Acc: 0.8989, F1: 0.9000, Time: 158.77s
      [+] Saved to Coffee_results.csv
      Generating 2x synthetic data with SMOTE...


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> [SMOTE @ 2x] Acc: 0.6933, F1: 0.6805, Time: 8.99s
      [+] Saved to Coffee_results.csv
      Generating 5x synthetic data with SMOTE...


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
